# 06. Final pipeline
Model choice from 02–05: **RT-DETR-L** has the best validation mAP under the same 1.25 GPU-hour budget
(0.297 vs 0.257 for the best CNN, YOLO11m at 1536) and the best public score (0.294).

In this notebook the model family is fixed and we change one thing at a time:
1. **Inference** (no training): horizontal-flip TTA and a WBF ensemble with YOLO11m-1536.
2. **Data** (same model and budget): masking of ignored regions, frame step.
3. **Temporal smoothing** (bonus): rescoring detections with neighbouring video frames.
4. **Final models** trained on train + val clips with the selected settings, and the test submissions.

All selection is done on the same validation frames as in 02–05 (420 frames from 7 held-out clips), except the
temporal step, which needs consecutive frames (section 3).

In [ ]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
%load_ext autoreload
%autoreload 2
import sys, json, pickle, subprocess, itertools
from pathlib import Path
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ultralytics import RTDETR, YOLO
from common import (PROC, ROOT, PROJECT, CHECKPOINTS, seed_everything, load_results, coco_evaluate, save_results,
                    wbf_merge, image_sizes, load_split, select_frames, build_yolo_dataset, make_coco_gt,
                    temporal_rescore, write_submission)
import yolo_utils as yu

seed_everything()
VAL_GT = PROC / "val_gt.json"
SIZES = image_sizes(VAL_GT)
ITEMS = yu.val_items(VAL_GT)
BUDGET_H = 1.25
RT_CFG = dict(model="rtdetr-l", queries=300, denoising=100, budget_h=BUDGET_H, batch=1)
KEYS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]
COMP = "multiple-object-video-detection-hw-2-ucu-ai-2026"
SUBMIT = True          # set to False to only write the csv files without sending them to Kaggle

def load_preds(name):
    return pickle.load(open(CHECKPOINTS / f"{name}_val_preds.pkl", "rb"))

def cached(path, fn):
    """Compute predictions once and keep them in checkpoints/ (inference on 400+ frames takes minutes)."""
    path = CHECKPOINTS / path
    if path.exists():
        return pickle.load(open(path, "rb"))
    out = fn()
    pickle.dump(out, open(path, "wb"))
    return out

def submit(path, message):
    if SUBMIT:
        subprocess.run(["kaggle", "competitions", "submit", "-c", COMP, "-f", str(path), "-m", message], check=True)

## 1. Inference strategies (model fixed: `rtdetr_base`, trained on train clips only)
* **Flip TTA**: predict on the frame and on its horizontal mirror, map the mirrored boxes back, merge both sets.
* **Ensemble** with **YOLO11m-1536** (`yolo_1536`): a different family (CNN, anchor-free, NMS-based), so its errors
  should be less correlated with RT-DETR's; it was also the best CNN on medium objects.

Merging uses **Weighted Boxes Fusion** (Solovyev et al., 2019, package `ensemble-boxes`): boxes of the same class
that overlap above an IoU threshold are averaged (weighted by score) instead of suppressed. The grid below is kept
small on purpose: tuning many ensemble parameters on 7 validation clips would overfit them.

In [ ]:
rt_base = RTDETR(str(yu.RUNS / "rtdetr_base" / "weights" / "best.pt"))
P_VAL = {"rt": load_preds("rtdetr_base"),
         "rt_flip": cached("rtdetr_base_flip_val_preds.pkl", lambda: yu.predict_ultra_hflip(rt_base, ITEMS, imgsz=1280)),
         "yolo": load_preds("yolo_1536")}

def apply_recipe(P, sizes, flip, yolo, w_rt=2, iou=0.6):
    """One inference recipe: RT-DETR alone or merged (WBF) with its flipped pass and/or YOLO11m-1536."""
    parts, weights = [P["rt"]], [w_rt]
    if flip:
        parts.append(P["rt_flip"]); weights.append(w_rt)
    if yolo:
        parts.append(P["yolo"]); weights.append(1)
    return P["rt"] if len(parts) == 1 else wbf_merge(parts, sizes, weights=weights, iou_thr=iou)

grid = [dict(flip=False, yolo=False)]
grid += [dict(flip=True, yolo=False, iou=t) for t in (0.55, 0.65)]
grid += [dict(flip=f, yolo=True, w_rt=w, iou=t) for f in (False, True) for w in (1, 2, 3) for t in (0.55, 0.65)]
rows = []
for g in grid:
    m = coco_evaluate(VAL_GT, apply_recipe(P_VAL, SIZES, **g))
    rows.append({**g, **{k: m[k] for k in KEYS}})
inf_tab = pd.DataFrame(rows).sort_values("mAP", ascending=False)
display(inf_tab.round(4))

BEST_RECIPE = {k: v for k, v in inf_tab.iloc[0][["flip", "yolo", "w_rt", "iou"]].items() if pd.notna(v)}
BEST_RECIPE = {k: (bool(v) if k in ("flip", "yolo") else float(v) if k == "iou" else int(v)) for k, v in BEST_RECIPE.items()}
print("selected inference recipe:", BEST_RECIPE)
best_val = apply_recipe(P_VAL, SIZES, **BEST_RECIPE)
save_results("final_inference_recipe_val", coco_evaluate(VAL_GT, best_val), recipe=str(BEST_RECIPE),
             models="rtdetr_base + yolo_1536 (trained on train clips)")

## 2. Data strategies (model and budget fixed: RT-DETR-L, 1.25 h)
* **No masking** (`s10_nomask`): ignored regions are not painted grey and objects inside them are kept.
  Tests the preprocessing choice from 01_data.
* **Frame step 5** (`s5`): twice as many frames from the same 56 clips. Tests whether the model is limited by the
  number of frames or by the number of distinct scenes (02–05 all peaked early, which suggests the latter).

In [ ]:
imgs_tr, obj_tr, ign_tr = load_split("train")
imgs_va, obj_va, ign_va = load_split("val")
val_frames = select_frames(imgs_va, per_clip=60)            # the same validation frames as everywhere
for name, step, mask in [("s10_nomask", 10, False), ("s5", 5, True)]:
    if not (PROC / name / "data.yaml").exists():
        tr = select_frames(imgs_tr, step=step)
        st = build_yolo_dataset(name, {"train": ("train", tr, obj_tr, ign_tr, mask),
                                       "val": ("val", val_frames, obj_va, ign_va, False)})
        print(name, st.groupby("part")[["kept", "dropped"]].sum().to_dict())

In [ ]:
# ~1.25 h
w_nomask = yu.train_ultra("rtdetr_nomask", "rtdetr-l.pt", kind="rtdetr", imgsz=1280, batch=1, hours=BUDGET_H,
                          data=str(PROC / "s10_nomask" / "data.yaml"))
res_nomask, _ = yu.evaluate_ultra("rtdetr_nomask", w_nomask, VAL_GT, {**RT_CFG, "data": "s10, no masking"},
                                  kind="rtdetr", imgsz=1280)

In [ ]:
# ~1.25 h
w_s5 = yu.train_ultra("rtdetr_s5", "rtdetr-l.pt", kind="rtdetr", imgsz=1280, batch=1, hours=BUDGET_H,
                      data=str(PROC / "s5" / "data.yaml"))
res_s5, _ = yu.evaluate_ultra("rtdetr_s5", w_s5, VAL_GT, {**RT_CFG, "data": "s5 (every 5th frame)"},
                              kind="rtdetr", imgsz=1280)

data_tab = load_results().set_index("name").loc[["rtdetr_base", "rtdetr_nomask", "rtdetr_s5"], KEYS + ["epochs_done"]]
display(data_tab)
DATA_OPTIONS = {"rtdetr_base": (10, True), "rtdetr_nomask": (10, False), "rtdetr_s5": (5, True)}
BEST_DATA = data_tab["mAP"].idxmax()
TRAIN_STEP, MASK = DATA_OPTIONS[BEST_DATA]
print(f"selected data setting: {BEST_DATA} -> frame step {TRAIN_STEP}, masking {MASK}")

## 3. Temporal smoothing (bonus: data temporality)
Test frames are consecutive video frames, but our validation frames are 60 *spaced* frames per clip. To evaluate a
temporal method we therefore build a second validation set: **60 consecutive frames from the middle of every
validation clip** (same 7 held-out clips, models trained on train clips only).

Method (`common.temporal_rescore`, in the spirit of Seq-NMS / T-CNN tubelet rescoring): for every detection we look
at the frames t±1..t±window and take the score of the best matching detection there (same class, IoU >= 0.5).
New score = alpha * own score + (1 - alpha) * mean neighbour score. Objects that are detected consistently keep
their score; isolated false positives lose it. Boxes are not changed, so this only affects the ranking.

In [ ]:
# consecutive validation segment: 60 frames from the middle of each validation clip
seq = []
for clip, g in imgs_va.sort_values("frame_id").groupby("video"):
    mid = len(g) // 2
    seq.append(g.iloc[max(0, mid - 30): mid + 30])
seq_frames = pd.concat(seq, ignore_index=True)
SEQ_GT = make_coco_gt(seq_frames, obj_va, ign_va, "val", PROC / "val_seq_gt.json")
SEQ_ITEMS, SEQ_SIZES = yu.val_items(SEQ_GT), image_sizes(SEQ_GT)
SEQ_FRAME_OF = {r.image_id: (r.video, r.frame_id) for r in seq_frames.itertuples()}
print(len(seq_frames), "consecutive frames from", seq_frames.video.nunique(), "clips")

yolo_1536 = YOLO(str(yu.RUNS / "yolo_1536" / "weights" / "best.pt"))
P_SEQ = {"rt": cached("rtdetr_base_seq_preds.pkl", lambda: yu.predict_ultra(rt_base, SEQ_ITEMS, imgsz=1280)[0]),
         "rt_flip": cached("rtdetr_base_flip_seq_preds.pkl", lambda: yu.predict_ultra_hflip(rt_base, SEQ_ITEMS, imgsz=1280)),
         "yolo": cached("yolo_1536_seq_preds.pkl", lambda: yu.predict_ultra(yolo_1536, SEQ_ITEMS, imgsz=1536)[0])}

In [ ]:
seq_single = P_SEQ["rt"]
seq_recipe = apply_recipe(P_SEQ, SEQ_SIZES, **BEST_RECIPE)
rows = [{"setting": "rtdetr single pass", **coco_evaluate(SEQ_GT, seq_single)},
        {"setting": f"inference recipe {BEST_RECIPE}", **coco_evaluate(SEQ_GT, seq_recipe)}]
for window, alpha in itertools.product([1, 2, 3], [0.4, 0.6, 0.8]):
    m = coco_evaluate(SEQ_GT, temporal_rescore(seq_recipe, SEQ_FRAME_OF, window=window, alpha=alpha))
    rows.append({"setting": f"recipe + temporal (window={window}, alpha={alpha})", "window": window, "alpha": alpha, **m})
temp_tab = pd.DataFrame(rows)
display(temp_tab[["setting"] + KEYS].round(4))

best_t = temp_tab.dropna(subset=["window"]).sort_values("mAP", ascending=False).iloc[0]
USE_TEMPORAL = best_t["mAP"] > temp_tab.loc[1, "mAP"]
TEMPORAL = dict(window=int(best_t["window"]), alpha=float(best_t["alpha"])) if USE_TEMPORAL else None
print("temporal smoothing:", TEMPORAL if USE_TEMPORAL else "not used (no gain on the consecutive validation frames)")

## 4. Final models: train + validation clips
The model is limited by the number of distinct scenes (02–05 peak early; section 2), so the final models are trained
on all 63 labelled clips (56 train + 7 val) with the selected data setting and the same 1.25 h budget.
There is no held-out data left for them, so (1) every choice was made above with models trained on train clips only,
and (2) we take the **last** checkpoint instead of the "best" one: Ultralytics' best-epoch selection would now use
frames the model was trained on. This is safe here because RT-DETR's validation curve was flat after ~0.5 h (05).

In [ ]:
full_name = f"full_s{TRAIN_STEP}{'' if MASK else '_nomask'}"
if not (PROC / full_name / "data.yaml").exists():
    tr = select_frames(imgs_tr, step=TRAIN_STEP)
    va = select_frames(imgs_va, step=TRAIN_STEP)
    st = build_yolo_dataset(full_name, {"train": ("train", tr, obj_tr, ign_tr, MASK),
                                        "train_val": ("val", va, obj_va, ign_va, MASK),
                                        "val": ("val", val_frames, obj_va, ign_va, False)})
    print(st.groupby("part")[["kept", "dropped"]].sum())
# both labelled parts are used for training; "val" is only Ultralytics' progress monitor (it is inside train now)
yaml_path = PROC / full_name / "data.yaml"
yaml_path.write_text(yaml_path.read_text().replace("train: images/train\n", "train: [images/train, images/train_val]\n"))
FULL_YAML = str(yaml_path)
print(yaml_path.read_text())

In [ ]:
# ~1.25 h
yu.train_ultra("rtdetr_final", "rtdetr-l.pt", kind="rtdetr", imgsz=1280, batch=1, hours=BUDGET_H, data=FULL_YAML)
W_RT_FINAL = yu.RUNS / "rtdetr_final" / "weights" / "last.pt"

In [ ]:
# ~1.25 h, only needed if the selected recipe uses YOLO
if BEST_RECIPE.get("yolo"):
    yu.train_ultra("yolo_final", "yolo11m.pt", imgsz=1536, batch=2, hours=BUDGET_H, data=FULL_YAML)
    W_YOLO_FINAL = yu.RUNS / "yolo_final" / "weights" / "last.pt"

## 5. Test predictions and submissions
Three submissions, from simple to full, so the public score shows the effect of each step:
* **A**: final RT-DETR alone;
* **B**: final RT-DETR + selected inference recipe (TTA / ensemble);
* **C**: B + temporal smoothing (if it helped in section 3).

In [ ]:
test_df = pd.read_csv(ROOT / "test_images.csv")
T_ITEMS = yu.test_items(test_df)
T_SIZES = {r.image_id: (r.width, r.height) for r in test_df.itertuples()}
T_FRAME_OF = {r.image_id: (r.clip, r.frame) for r in test_df.itertuples()}

rt_final = RTDETR(str(W_RT_FINAL))
P_TEST = {"rt": cached("rtdetr_final_test_preds.pkl", lambda: yu.predict_ultra(rt_final, T_ITEMS, imgsz=1280)[0])}
if BEST_RECIPE.get("flip"):
    P_TEST["rt_flip"] = cached("rtdetr_final_flip_test_preds.pkl", lambda: yu.predict_ultra_hflip(rt_final, T_ITEMS, imgsz=1280))
if BEST_RECIPE.get("yolo"):
    yolo_final = YOLO(str(W_YOLO_FINAL))
    P_TEST["yolo"] = cached("yolo_final_test_preds.pkl", lambda: yu.predict_ultra(yolo_final, T_ITEMS, imgsz=1536)[0])

In [ ]:
SUB = PROJECT / "submissions"
sub_a = P_TEST["rt"]
write_submission(sub_a, test_df, SUB / "final_A_rtdetr.csv")
submit(SUB / "final_A_rtdetr.csv", "Final A: RT-DETR-L trained on train+val")

sub_b = apply_recipe(P_TEST, T_SIZES, **BEST_RECIPE)
write_submission(sub_b, test_df, SUB / "final_B_recipe.csv")
submit(SUB / "final_B_recipe.csv", f"Final B: RT-DETR train+val + {BEST_RECIPE}")

if USE_TEMPORAL:
    sub_c = temporal_rescore(sub_b, T_FRAME_OF, **TEMPORAL)
    write_submission(sub_c, test_df, SUB / "final_C_temporal.csv")
    submit(SUB / "final_C_temporal.csv", f"Final C: B + temporal {TEMPORAL}")

## 6. Summary of all models
Validation metrics of every run from 02–06 (same 420 frames, same evaluator). Speed is model time per frame on an
RTX 5070 Laptop GPU (torchvision: mixed precision, Ultralytics: inference + NMS, both at batch 1).

In [ ]:
res = load_results().set_index("name")
family = {"frcnn": "two-stage anchor-based", "retina": "one-stage anchor-based", "yolo": "one-stage anchor-free",
          "rtdetr": "query-based"}
res["family"] = [next((v for k, v in family.items() if n.startswith(k)), "final pipeline") for n in res.index]
cols = ["family", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms_per_image"]
summary = res[cols].sort_values(["family", "mAP"], ascending=[True, False])
display(summary.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
for fam, g in summary.dropna(subset=["ms_per_image"]).groupby("family"):
    ax.scatter(g.ms_per_image, g.mAP, label=fam, s=40)
    for n, r in g.iterrows():
        ax.annotate(n, (r.ms_per_image, r.mAP), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("ms per frame"); ax.set_ylabel("validation mAP"); ax.legend(fontsize=8)
ax.set_title("Accuracy vs speed of all trained models"); plt.tight_layout(); plt.show()